In [ ]:
from influxdb import InfluxDBClient
import random
import numpy as np
import time
import matplotlib.pyplot as plt
from dateutil.parser import parse

host = '149.62.71.186'
user = 'admin'
password = 'fis_influx'
port=8086
client = InfluxDBClient(host, port, user, password)

In [ ]:
client.switch_database('Naloga 0')

# (a)

In [ ]:
def parse_result_a(result,field, wanted_tag_type,wanted_tag_value):
    #we need to parse the result of the query
    time, T1 = [],[]
    for (measurement_name, tags), points in result.items():
        tag_type = list(tags.keys())[0]
        tag_value = tags[list(tags.keys())[0]]
        print(f'tag type:{tag_type}, tag_value:{tag_value}')
        if(tag_value != wanted_tag_value or tag_type!= wanted_tag_type):
            continue
        print(f'reading data  with tag: {tags}')
        for point in points:
            timestamp = point.get('time')
            t1 = point.get(field)
            tags = point.get('user')
            T1.append(t1)
            time.append(timestamp)
    return time, T1


def narisi_od_tag_a(database, measurement, field, tag):
    tag_type = list(tag.keys())[0]
    tag_value = tag[list(tag.keys())[0]]
    res=client.query(f'SELECT "{field}" FROM "{database}".."{measurement}" GROUP BY "{tag_type}"')
    
    time, T1 = parse_result_a(res,field, tag_type, tag_value)
    #print(res)
    
    datetime_objects = [parse(date) for date in time]
    fig, ax = plt.subplots()
    ax.xaxis.set_tick_params(rotation=90)
    plt.plot(datetime_objects,T1, label='T1')
    plt.legend()

narisi_od_tag_a('Naloga 0', 'Temperature', 'T1', {'user' : 'user2'})   

In [ ]:
#alternativna rešitev
from datetime import datetime

field = 'T1'
measurement = 'Temperature'
database = 'Naloga 0'
tag_type = 'user'
tag_value = 'user1'

def narisi_od_tag_a(database, measurement, field, tag):
    tag_type = list(tag.keys())[0]
    tag_value = tag[tag_type]
    result=client.query(f'SELECT "{field}" FROM "{database}".."{measurement}" GROUP BY "{tag_type}"')

    items = result.items()

    for item in items:
        descripiton, generator =  item
        meas, tag_ = descripiton
        tag_type_ = list(tag_.keys())[0]
        tag_value_ = tag_[tag_type]
        if tag_value != tag_value_:
            continue

        values, times = [],[]
        for point in generator:
                times.append(point['time'])
                values.append(point['T1'])

        #turn to seconds     
        dt = [datetime.fromisoformat(i.replace('Z', '+00:00')).timestamp() for i in times]
        #start from zero time
        dt_zero = [i-dt[0] for i in dt]

        plt.figure(figsize = (12,4))
        plt.plot(dt_zero,values)

narisi_od_tag_a(database, measurement, field, {tag_type:tag_value})  

# (b)

In [ ]:
def parse_result_b(result,field, wanted_tag_type,wanted_tag_value):
    #we need to parse the result of the query
    time, T1 = [],[]
    for (measurement_name, tags), points in result.items():
        for point in points:
            timestamp = point.get('time')
            t1 = point.get(field)
            tags = point.get(wanted_tag_type)
            if(tags!=wanted_tag_value):
                continue
            T1.append(t1)
            time.append(timestamp)
    return time, T1


def narisi_od_tag_b(database, measurement, field, tag):
    
    tag_type = list(tag.keys())[0]
    tag_value = tag[list(tag.keys())[0]]
    
    res=client.query(f'SELECT * FROM "{database}".."{measurement}"')
    
    time, T1 = parse_result_b(res,field, tag_type, tag_value)
    
    datetime_objects = [parse(date) for date in time]
    fig, ax = plt.subplots()
    ax.xaxis.set_tick_params(rotation=90)
    plt.plot(datetime_objects,T1, label='T1')
    plt.legend()

narisi_od_tag_b('Naloga 0', 'Temperature', 'T1', {'user' : 'user1'})  

In [ ]:
#alternativna rešitev
from datetime import datetime

field = 'T1'
measurement = 'Temperature'
database = 'Naloga 0'
tag_type = 'user'
tag_value = 'user1'

def narisi_od_tag_b(database, measurement, field, tag):
    tag_type = list(tag.keys())[0]
    tag_value = tag[tag_type]
    result=client.query(f'SELECT * FROM "{database}".."{measurement}"')

    items = result.items()

    for item in items:
        descripiton, generator =  item
        meas, tag_ = descripiton
        values, times = [],[]
        for point in generator:
            if(point[tag_type]!=tag_value):
                continue
            times.append(point['time'])
            values.append(point['T1'])

        #turn to seconds     
        dt = [datetime.fromisoformat(i.replace('Z', '+00:00')).timestamp() for i in times]
        #start from zero time
        dt_zero = [i-dt[0] for i in dt]

        plt.figure(figsize = (12,4))
        plt.plot(dt_zero,values)

narisi_od_tag_b(database, measurement, field, {tag_type:tag_value})  

# (C)

In [ ]:
def parse_result_c(result,field, wanted_tag_type,wanted_tag_value):
    #we need to parse the result of the query
    time, T1 = [],[]
    for (measurement_name, tags), points in result.items():
        for point in points:
            timestamp = point.get('time')
            t1 = point.get(field)
            tags = point.get(wanted_tag_type)
            print(tags)
            T1.append(t1)
            time.append(timestamp)
    return time, T1


def narisi_od_tag_c(database, measurement, field, tag):
    tag_type = list(tag.keys())[0]
    tag_value = tag[list(tag.keys())[0]]
    
    query = f'SELECT "{field}" FROM "{database}".."{measurement}" WHERE "{tag_type}"=\'{tag_value}\''
    #query = f'SELECT "{field}" FROM "{database}".."{measurement}" WHERE "user"=\'Ziga\''
    print(query)
    res=client.query(query)
    
    time, T1 = parse_result_c(res,field, tag_type, tag_value)
    #print(res)
    
    datetime_objects = [parse(date) for date in time]
    fig, ax = plt.subplots()
    ax.xaxis.set_tick_params(rotation=90)
    plt.plot(datetime_objects,T1, label='T1')
    plt.legend()

narisi_od_tag_c('Naloga 0', 'Temperature', 'T1', {'user' : 'user1'}) 

In [ ]:
#alternativna rešitev
from datetime import datetime

field = 'T1'
measurement = 'Temperature'
database = 'Naloga 0'
tag_type = 'user'
tag_value = 'user1'

def narisi_od_tag_c(database, measurement, field, tag):
    tag_type = list(tag.keys())[0]
    tag_value = tag[tag_type]
    result =client.query(f'SELECT "{field}" FROM "{database}".."{measurement}" WHERE "{tag_type}"=\'{tag_value}\'')
    items = result.items()

    for item in items:
        descripiton, generator =  item
        meas, tag_ = descripiton
        values, times = [],[]
        for point in generator:
            times.append(point['time'])
            values.append(point['T1'])

        #turn to seconds     
        dt = [datetime.fromisoformat(i.replace('Z', '+00:00')).timestamp() for i in times]
        #start from zero time
        dt_zero = [i-dt[0] for i in dt]

        plt.figure(figsize = (12,4))
        plt.plot(dt_zero,values)

narisi_od_tag_c(database, measurement, field, {tag_type:tag_value})  